# Customer Sales Performance Analysis and Business Insights Dashboard
**End-to-end Data Analytics Assignment**

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** Sample Superstore Dataset (CSV)

> Before running: place `Sample - Superstore.csv` (or your Superstore CSV) in the same folder as this notebook. If your file has a different name, update `DATA_PATH` in Phase 2.

## Phase 1 — Business Understanding

### Problem statement
The retail company wants to understand sales performance, profit variation, customer behavior, and product performance using historical sales data.

### Business objectives
1. Identify high-revenue products.
2. Find profitable and unprofitable regions.
3. Understand valuable customer segments.
4. Analyze sales and profit trends.
5. Provide business recommendations.

### KPIs
- Total Sales
- Total Profit
- Total Quantity
- Average Order Value (AOV)
- Profit Margin
- Sales and profit by category, region, segment, and time

### Risks and success criteria
Risks include missing or incorrect data, outliers, misleading comparisons, and discount-driven losses. Success means producing reliable summaries, useful visualizations, and actionable recommendations.

## Phase 2 — Data Loading and Inspection

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.ticker import FuncFormatter

# Professional visual theme
PALETTE = ["#2563EB", "#7C3AED", "#06B6D4", "#F59E0B",
           "#10B981", "#EF4444", "#EC4899", "#64748B"]
BG = "#F7F9FC"
TEXT = "#243247"
GRID = "#DCE3ED"

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette=PALETTE,
    rc={
        "figure.facecolor": BG,
        "axes.facecolor": "white",
        "axes.edgecolor": "#E5EAF1",
        "axes.labelcolor": TEXT,
        "text.color": TEXT,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "grid.color": GRID,
        "grid.linestyle": "--",
        "grid.alpha": 0.55,
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "font.family": "DejaVu Sans",
        "figure.titlesize": 20,
        "figure.titleweight": "bold"
    }
)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    if title:
        ax.set_title(title, loc="left", pad=14, fontsize=14, fontweight="bold", color=TEXT)
    if xlabel is not None:
        ax.set_xlabel(xlabel, labelpad=8)
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=8)
    ax.grid(axis="y", alpha=0.45, linestyle="--")
    ax.set_axisbelow(True)
    ax.spines["left"].set_color("#DCE3ED")
    ax.spines["bottom"].set_color("#DCE3ED")
    return ax

def format_currency_axis(ax, axis="y"):
    formatter = FuncFormatter(lambda x, pos: f"${x:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(formatter)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "Sample - Superstore.csv"  # Change this if your CSV has another name

df = pd.read_csv(DATA_PATH, encoding="latin1")
print("First five rows:")
display(df.head())

print("\nDataset shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nMemory usage (KB):", round(df.memory_usage(deep=True).sum() / 1024, 2))
print("\nDataset information:")
df.info()

In [2]:
overview = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing Values", "Duplicate Records",
               "Numerical Columns", "Categorical Columns", "Date Columns",
               "Memory Usage (KB)"],
    "Value": [
        df.shape[0], df.shape[1], int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        len(df.select_dtypes(include=np.number).columns),
        len(df.select_dtypes(include="object").columns),
        len(df.select_dtypes(include=["datetime", "datetimetz"]).columns),
        round(df.memory_usage(deep=True).sum() / 1024, 2)
    ]
})
display(overview)

,Metric,Value
0,Rows,"9,994.00"
1,Columns,21.00
2,Missing Values,0.00
3,Duplicate Records,0.00
4,Numerical Columns,6.00
5,Categorical Columns,15.00
6,Date Columns,0.00
7,Memory Usage (KB),"10,616.54"


## Phase 3 — Data Cleaning

This section checks missing values, duplicate rows, data types, and outliers. The assignment's supplied report states that the 9,994-row dataset has no missing values or duplicate records, and that extreme values were retained after review because they represent real transactions.

In [3]:
# Standardize column labels by removing leading/trailing spaces
df.columns = df.columns.str.strip()

# Convert Order Date to datetime if the column exists
if "Order Date" in df.columns:
    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

# Convert Ship Date if available
if "Ship Date" in df.columns:
    df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

print("Missing values by column:")
display(df.isna().sum().to_frame("Missing Values"))

print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows only
df = df.drop_duplicates().copy()

print("\nData types after conversion:")
display(df.dtypes)

# IQR outlier counts for numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns
outlier_rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append([col, lower, upper, int(count)])

outlier_report = pd.DataFrame(
    outlier_rows, columns=["Column", "Lower Bound", "Upper Bound", "Outlier Count"]
)
display(outlier_report)

# Keep extreme values unless investigation shows they are data errors.
clean_df = df.copy()

Missing values by column:


,Missing Values
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


Duplicate rows: 0

Data types after conversion:


Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code               int64
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

,Column,Lower Bound,Upper Bound,Outlier Count
0,Row ID,"-4,995.50","14,990.50",0
1,Postal Code,"-76,954.50","190,185.50",0
2,Sales,-271.71,498.93,1167
3,Quantity,-2.50,9.50,170
4,Discount,-0.30,0.50,856
5,Profit,-39.72,70.82,1881


## Phase 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive statistics
display(clean_df.describe(include="all").T)

# Attractive univariate distributions
for col, color in zip(
    ["Sales", "Profit", "Quantity", "Discount"],
    PALETTE
):
    if col in clean_df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        fig.suptitle(f"{col} — Distribution Overview", x=0.08, ha="left",
                     fontsize=17, fontweight="bold", color=TEXT)

        sns.histplot(
            clean_df[col].dropna(), kde=True, ax=axes[0],
            color=color, edgecolor="white", alpha=0.85,
            line_kws={"linewidth": 2.5}
        )
        style_ax(axes[0], f"{col} Distribution", xlabel=col, ylabel="Frequency")

        sns.boxplot(
            x=clean_df[col], ax=axes[1], color=color, width=0.42,
            flierprops={"marker": "o", "markersize": 3, "alpha": 0.35}
        )
        style_ax(axes[1], f"{col} Box Plot", xlabel=col)
        plt.tight_layout(rect=[0, 0, 1, 0.90])
        plt.show()

# Bivariate and multivariate overview
selected = [c for c in ["Sales", "Profit", "Quantity", "Discount"] if c in clean_df.columns]
pair_data = clean_df[selected].dropna()
if len(pair_data) > 1500:
    pair_data = pair_data.sample(1500, random_state=42)

pair = sns.pairplot(
    pair_data, diag_kind="kde", corner=True,
    plot_kws={"alpha": 0.45, "s": 22, "edgecolor": "none"},
    diag_kws={"fill": True, "alpha": 0.55},
    palette=PALETTE
)
pair.fig.suptitle("Relationships Between Key Business Metrics",
                  y=1.02, fontsize=17, fontweight="bold", color=TEXT)
plt.show()

## Phase 5 — Category Analysis

In [ ]:
category_summary = clean_df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).sort_values("Total_Sales", ascending=False)

display(category_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Category Performance", x=0.06, ha="left",
             fontsize=19, fontweight="bold", color=TEXT)

sales_colors = sns.color_palette("Blues", n_colors=len(category_summary) + 2)[2:]
profit_colors = [("#10B981" if v >= 0 else "#EF4444")
                 for v in category_summary["Total_Profit"]]

bars = axes[0].bar(category_summary.index, category_summary["Total_Sales"],
                   color=sales_colors, width=0.62, edgecolor="white", linewidth=1.2)
style_ax(axes[0], "Total Sales by Category", ylabel="Sales")
format_currency_axis(axes[0])
axes[0].bar_label(bars, labels=[f"${v:,.0f}" for v in category_summary["Total_Sales"]],
                  padding=4, fontsize=9, fontweight="bold")
axes[0].tick_params(axis="x", rotation=12)

bars = axes[1].bar(category_summary.index, category_summary["Total_Profit"],
                   color=profit_colors, width=0.62, edgecolor="white", linewidth=1.2)
style_ax(axes[1], "Total Profit by Category", ylabel="Profit")
format_currency_axis(axes[1])
axes[1].bar_label(bars, labels=[f"${v:,.0f}" for v in category_summary["Total_Profit"]],
                  padding=4, fontsize=9, fontweight="bold")
axes[1].tick_params(axis="x", rotation=12)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 6))
wedges, texts, autotexts = ax.pie(
    category_summary["Total_Sales"],
    labels=category_summary.index,
    autopct="%1.1f%%", startangle=90,
    colors=PALETTE[:len(category_summary)],
    pctdistance=0.72,
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2},
    textprops={"color": TEXT, "fontsize": 10}
)
for t in autotexts:
    t.set_fontweight("bold")
    t.set_color("white")
ax.set_title("Sales Contribution by Category", loc="left",
             fontsize=16, fontweight="bold", pad=15)
ax.text(0, 0, "SALES\\nSHARE", ha="center", va="center",
        fontsize=12, fontweight="bold", color=TEXT)
plt.tight_layout()
plt.show()

## Phase 6 — Regional Analysis

In [ ]:
region_summary = clean_df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Sales", ascending=False)

display(region_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Regional Performance", x=0.06, ha="left",
             fontsize=19, fontweight="bold", color=TEXT)

region_colors = sns.color_palette("viridis", n_colors=len(region_summary))
bars = axes[0].bar(region_summary.index, region_summary["Total_Sales"],
                   color=region_colors, width=0.65, edgecolor="white")
style_ax(axes[0], "Sales by Region", ylabel="Sales")
format_currency_axis(axes[0])
axes[0].bar_label(bars, labels=[f"${v:,.0f}" for v in region_summary["Total_Sales"]],
                  padding=3, fontsize=8, fontweight="bold")
axes[0].tick_params(axis="x", rotation=10)

profit_colors = ["#10B981" if x >= 0 else "#EF4444"
                 for x in region_summary["Total_Profit"]]
bars = axes[1].bar(region_summary.index, region_summary["Total_Profit"],
                   color=profit_colors, width=0.65, edgecolor="white")
style_ax(axes[1], "Profit by Region", ylabel="Profit")
format_currency_axis(axes[1])
axes[1].bar_label(bars, labels=[f"${v:,.0f}" for v in region_summary["Total_Profit"]],
                  padding=3, fontsize=8, fontweight="bold")
axes[1].tick_params(axis="x", rotation=10)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

region_category = clean_df.pivot_table(
    index="Region", columns="Category", values="Profit", aggfunc="sum"
)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    region_category, annot=True, fmt=",.0f", cmap="YlGnBu",
    linewidths=1.2, linecolor="white", cbar_kws={"label": "Total Profit"},
    annot_kws={"fontsize": 10, "fontweight": "bold"}, ax=ax
)
style_ax(ax, "Profit Heatmap — Region vs Category")
ax.set_xlabel("Product Category")
ax.set_ylabel("Region")
plt.tight_layout()
plt.show()

## Phase 7 — Customer Analysis

In [ ]:
customer_summary = clean_df.groupby("Customer Name").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

print("Top 10 customers by revenue:")
display(customer_summary.sort_values("Total_Sales", ascending=False).head(10))

print("Top 10 customers by profit:")
display(customer_summary.sort_values("Total_Profit", ascending=False).head(10))

segment_summary = clean_df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
)
segment_summary["Sales_Contribution_%"] = (
    segment_summary["Total_Sales"] / segment_summary["Total_Sales"].sum() * 100
)
display(segment_summary)

segment_sales = segment_summary["Total_Sales"].sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(segment_sales.index, segment_sales.values,
               color=sns.color_palette("mako", n_colors=len(segment_sales)),
               edgecolor="white", height=0.58)
style_ax(ax, "Sales by Customer Segment", xlabel="Sales")
format_currency_axis(ax, axis="x")
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in segment_sales.values],
             padding=5, fontsize=9, fontweight="bold")
ax.grid(axis="x", alpha=0.4, linestyle="--")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## Phase 8 — Time Series Analysis

In [ ]:
# Create date-based features
clean_df["Order Month"] = clean_df["Order Date"].dt.month
clean_df["Order Year"] = clean_df["Order Date"].dt.year
clean_df["Order Quarter"] = clean_df["Order Date"].dt.to_period("Q").astype(str)

monthly_sales = clean_df.set_index("Order Date").resample("MS")["Sales"].sum()
quarterly_sales = clean_df.set_index("Order Date").resample("QS")["Sales"].sum()
yearly_sales = clean_df.set_index("Order Date").resample("YS")["Sales"].sum()

fig, axes = plt.subplots(3, 1, figsize=(14, 13))
fig.suptitle("Sales Performance Over Time", x=0.07, ha="left",
             fontsize=20, fontweight="bold", color=TEXT)

axes[0].plot(monthly_sales.index, monthly_sales.values, color="#2563EB",
             linewidth=2.8, marker="o", markersize=4)
axes[0].fill_between(monthly_sales.index, monthly_sales.values,
                     color="#2563EB", alpha=0.12)
style_ax(axes[0], "Monthly Sales Trend", ylabel="Sales")
format_currency_axis(axes[0])
axes[0].tick_params(axis="x", rotation=25)

axes[1].plot(quarterly_sales.index, quarterly_sales.values, color="#7C3AED",
             linewidth=2.8, marker="D", markersize=5)
axes[1].fill_between(quarterly_sales.index, quarterly_sales.values,
                     color="#7C3AED", alpha=0.12)
style_ax(axes[1], "Quarterly Sales Trend", ylabel="Sales")
format_currency_axis(axes[1])
axes[1].tick_params(axis="x", rotation=25)

year_labels = yearly_sales.index.strftime("%Y")
bars = axes[2].bar(year_labels, yearly_sales.values, color="#06B6D4",
                   width=0.58, edgecolor="white")
style_ax(axes[2], "Yearly Sales Comparison", ylabel="Sales")
format_currency_axis(axes[2])
axes[2].bar_label(bars, labels=[f"${v:,.0f}" for v in yearly_sales.values],
                  padding=4, fontsize=9, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

monthly_by_month = clean_df.groupby("Order Month")["Sales"].sum()
display(monthly_by_month)
print("Peak month number(s):", monthly_by_month[monthly_by_month == monthly_by_month.max()].index.tolist())
print("Lowest month number(s):", monthly_by_month[monthly_by_month == monthly_by_month.min()].index.tolist())

## Phase 9 — Profitability Analysis

In [ ]:
# Discount and profit relationship
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=clean_df, x="Discount", y="Profit",
    hue="Profit", palette="RdYlGn", center=0,
    alpha=0.62, s=48, linewidth=0, ax=ax,
    legend=False
)
ax.axhline(0, color="#334155", linewidth=1.2, linestyle="--", alpha=0.8)
style_ax(ax, "Discount vs Profit", xlabel="Discount", ylabel="Profit")
format_currency_axis(ax)
plt.tight_layout()
plt.show()

# Profit by product
product_summary = clean_df.groupby("Product Name").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Profit")

print("Lowest-profit products:")
display(product_summary.head(10))
print("Highest-profit products:")
display(product_summary.tail(10).sort_values("Total_Profit", ascending=False))

fig, ax = plt.subplots(figsize=(8, 6))
corr = clean_df[["Sales", "Profit", "Quantity", "Discount"]].corr()
sns.heatmap(
    corr, annot=True, cmap="vlag", center=0, fmt=".2f",
    square=True, linewidths=1.5, linecolor="white",
    annot_kws={"fontsize": 11, "fontweight": "bold"},
    cbar_kws={"shrink": 0.8, "label": "Correlation"}, ax=ax
)
style_ax(ax, "Correlation Between Business Metrics")
plt.tight_layout()
plt.show()

## Phase 10 — Statistical Analysis

In [10]:
stats_summary = clean_df[["Sales", "Profit", "Quantity"]].agg(
    ["mean", "median", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
     "var", "std"]
)
stats_summary.index = ["Mean", "Median", "Mode", "Variance", "Standard Deviation"]
display(stats_summary)

print("Correlation matrix:")
display(clean_df[["Sales", "Profit", "Quantity", "Discount"]].corr())

,Sales,Profit,Quantity
Mean,229.86,28.66,3.79
Median,54.49,8.67,3.00
Mode,12.96,0.00,3.00
Variance,"388,434.46","54,877.80",4.95
Standard Deviation,623.25,234.26,2.23


Correlation matrix:


,Sales,Profit,Quantity,Discount
Sales,1.00,0.48,0.20,-0.03
Profit,0.48,1.00,0.07,-0.22
Quantity,0.20,0.07,1.00,0.01
Discount,-0.03,-0.22,0.01,1.00


## Phase 11 — Feature Engineering

In [11]:
# Avoid division by zero
clean_df["Profit Margin"] = np.where(
    clean_df["Sales"] != 0, clean_df["Profit"] / clean_df["Sales"], np.nan
)
clean_df["Average Order Value"] = np.where(
    clean_df["Quantity"] != 0, clean_df["Sales"] / clean_df["Quantity"], np.nan
)

# Revenue bands based on sales distribution (tertiles)
clean_df["Revenue Category"] = pd.qcut(
    clean_df["Sales"].rank(method="first"),
    q=3,
    labels=["Low Revenue", "Medium Revenue", "High Revenue"]
)

display(clean_df[[
    "Sales", "Profit", "Quantity", "Profit Margin",
    "Average Order Value", "Revenue Category", "Order Month", "Order Year"
]].head())

print("Feature engineering completed.")

,Sales,Profit,Quantity,Profit Margin,Average Order Value,Revenue Category,Order Month,Order Year
0,261.96,41.91,2,0.16,130.98,High Revenue,11,2016
1,731.94,219.58,3,0.30,243.98,High Revenue,11,2016
2,14.62,6.87,2,0.47,7.31,Low Revenue,6,2016
3,957.58,-383.03,5,-0.40,191.52,High Revenue,10,2015
4,22.37,2.52,2,0.11,11.18,Low Revenue,10,2015


Feature engineering completed.


## Phase 12 — Advanced Pandas Analysis

In [12]:
# GroupBy and aggregation
business_summary = clean_df.groupby(["Category", "Region", "Segment"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Discount=("Discount", "mean"),
    Order_Count=("Order ID", "nunique")
).reset_index()
display(business_summary.head(20))

# Pivot table: monthly sales by category
monthly_category_pivot = clean_df.pivot_table(
    index="Order Month", columns="Category", values="Sales", aggfunc="sum"
)
display(monthly_category_pivot)

# Crosstab: region-wise segment contribution
region_segment_crosstab = pd.crosstab(
    clean_df["Region"], clean_df["Segment"], normalize="index"
) * 100
display(region_segment_crosstab.round(2))

# Multi-level grouping
display(clean_df.groupby(["Category", "Sub-Category"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Profit"))

,Category,Region,Segment,Total_Sales,Total_Profit,Average_Discount,Order_Count
0,Furniture,Central,Consumer,"86,229.22","-3,994.43",0.31,209
1,Furniture,Central,Corporate,"52,085.60","2,321.99",0.28,118
2,Furniture,Central,Home Office,"25,482.34","-1,198.61",0.28,76
3,Furniture,East,Consumer,"114,211.80","2,038.11",0.16,247
4,Furniture,East,Corporate,"64,209.05",342.13,0.15,160
5,Furniture,East,Home Office,"29,870.36",665.93,0.16,81
6,Furniture,South,Consumer,"70,800.20","4,616.73",0.11,150
7,Furniture,South,Corporate,"29,645.03",952.03,0.15,88
8,Furniture,South,Home Office,"16,853.45","1,202.44",0.10,40
9,Furniture,West,Consumer,"119,808.09","4,330.67",0.13,311


Category,Furniture,Office Supplies,Technology
Order Month,,,
1,"31,569.24","33,233.83","30,121.76"
2,"15,765.92","20,641.91","23,343.42"
3,"50,768.28","56,385.64","97,851.57"
4,"40,698.58","49,433.27","47,630.28"
5,"48,365.13","43,021.53","63,642.15"
6,"52,999.46","51,414.63","48,304.58"
7,"49,377.01","43,007.06","54,854.03"
8,"44,884.04","62,134.16","52,025.86"
9,"106,380.59","101,888.65","99,380.71"


Segment,Consumer,Corporate,Home Office
Region,,,
Central,52.17,28.97,18.85
East,51.58,30.79,17.63
South,51.73,31.48,16.79
West,52.20,29.97,17.83


Total_Sales  Total_Profit
Category        Sub-Category                           
Furniture       Tables         206,965.53    -17,725.48
                Bookcases      114,880.00     -3,472.56
Office Supplies Supplies        46,673.54     -1,189.10
                Fasteners        3,024.28        949.52
Technology      Machines       189,238.63      3,384.76
Office Supplies Labels          12,486.31      5,546.25
                Art             27,118.79      6,527.79
                Envelopes       16,476.40      6,964.18
Furniture       Furnishings     91,705.16     13,059.14
Office Supplies Appliances     107,532.16     18,138.01
                Storage        223,843.61     21,278.83
Furniture       Chairs         328,449.10     26,590.17
Office Supplies Binders        203,412.73     30,221.76
                Paper           78,479.21     34,053.57
Technology      Accessories    167,380.32     41,936.64
                Phones         330,007.05     44,515.73
                Copiers        149,528.03     55,617.82

## Phase 13 — Visualization Dashboard

In [ ]:
# Executive dashboard — consistent styling and clear visual hierarchy
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor(BG)
fig.suptitle("SUPERSTORE | BUSINESS PERFORMANCE DASHBOARD",
             x=0.06, y=0.99, ha="left", fontsize=21,
             fontweight="bold", color=TEXT)

# 1. Sales by category
cat_sales = clean_df.groupby("Category")["Sales"].sum().sort_values()
bars = axes[0, 0].barh(cat_sales.index, cat_sales.values,
                       color=sns.color_palette("Blues", len(cat_sales) + 2)[2:],
                       edgecolor="white")
style_ax(axes[0, 0], "Sales by Category", xlabel="Sales")
format_currency_axis(axes[0, 0], axis="x")
axes[0, 0].bar_label(bars, labels=[f"${v:,.0f}" for v in cat_sales.values],
                     padding=3, fontsize=8)

# 2. Profit by region
reg_profit = clean_df.groupby("Region")["Profit"].sum().sort_values()
reg_colors = ["#EF4444" if v < 0 else "#10B981" for v in reg_profit.values]
bars = axes[0, 1].bar(reg_profit.index, reg_profit.values,
                      color=reg_colors, edgecolor="white", width=0.65)
style_ax(axes[0, 1], "Profit by Region", ylabel="Profit")
format_currency_axis(axes[0, 1])
axes[0, 1].bar_label(bars, labels=[f"${v:,.0f}" for v in reg_profit.values],
                     padding=3, fontsize=8)
axes[0, 1].tick_params(axis="x", rotation=12)

# 3. Monthly sales trend
axes[0, 2].plot(monthly_sales.index, monthly_sales.values,
                color="#7C3AED", linewidth=2.6, marker="o", markersize=3)
axes[0, 2].fill_between(monthly_sales.index, monthly_sales.values,
                        color="#7C3AED", alpha=0.13)
style_ax(axes[0, 2], "Monthly Sales Trend", ylabel="Sales")
format_currency_axis(axes[0, 2])
axes[0, 2].tick_params(axis="x", rotation=35)

# 4. Top 10 customers
top_customers = clean_df.groupby("Customer Name")["Sales"].sum().nlargest(10).sort_values()
bars = axes[1, 0].barh(top_customers.index, top_customers.values,
                       color=sns.color_palette("crest", len(top_customers)),
                       edgecolor="white")
style_ax(axes[1, 0], "Top 10 Customers by Sales", xlabel="Sales")
format_currency_axis(axes[1, 0], axis="x")
axes[1, 0].tick_params(axis="y", labelsize=8)

# 5. Discount vs profit
sns.scatterplot(data=clean_df, x="Discount", y="Profit",
                hue="Profit", palette="RdYlGn", center=0,
                alpha=0.45, s=24, linewidth=0, legend=False,
                ax=axes[1, 1])
axes[1, 1].axhline(0, color="#334155", linestyle="--", linewidth=1)
style_ax(axes[1, 1], "Discount vs Profit", xlabel="Discount", ylabel="Profit")
format_currency_axis(axes[1, 1])

# 6. Correlation heatmap
corr = clean_df[["Sales", "Profit", "Quantity", "Discount"]].corr()
sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f",
            square=True, linewidths=1, linecolor="white",
            cbar=False, annot_kws={"fontsize": 9, "fontweight": "bold"},
            ax=axes[1, 2])
style_ax(axes[1, 2], "Metric Correlation")
axes[1, 2].tick_params(axis="x", rotation=30)
axes[1, 2].tick_params(axis="y", rotation=0)

for ax in axes.flat:
    ax.set_facecolor("white")
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    ax.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=3.0, w_pad=2.2)
fig.savefig("superstore_business_dashboard.png", dpi=300,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

## Phase 14 — Business Recommendations

In [14]:
# Create evidence-based recommendation lists from the calculated results
category_profit = clean_df.groupby("Category")["Profit"].sum().sort_values(ascending=False)
region_profit = clean_df.groupby("Region")["Profit"].sum().sort_values(ascending=False)
product_profit = clean_df.groupby("Product Name")["Profit"].sum().sort_values()

print("Categories ranked by profit:")
display(category_profit)

print("Regions ranked by profit:")
display(region_profit)

print("Products with the lowest total profit:")
display(product_profit.head(10))

print("Suggested actions to investigate:")
print("- Promote products with consistently high sales and positive profit.")
print("- Review products with repeated negative profit before discontinuing them.")
print("- Examine discount levels and their relationship with profit.")
print("- Investigate regions with relatively low profit.")
print("- Plan inventory and campaigns around observed peak sales periods.")

Categories ranked by profit:


Category
Technology        145,454.95
Office Supplies   122,490.80
Furniture          18,451.27
Name: Profit, dtype: float64

Regions ranked by profit:


Region
West      108,418.45
East       91,522.78
South      46,749.43
Central    39,706.36
Name: Profit, dtype: float64

Products with the lowest total profit:


Product Name
Cubify CubeX 3D Printer Double Head Print                           -8,879.97
Lexmark MX611dhe Monochrome Laser Printer                           -4,589.97
Cubify CubeX 3D Printer Triple Head Print                           -3,839.99
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2,876.12
Bush Advantage Collection Racetrack Conference Table                -1,934.40
GBC DocuBind P400 Electric Binding System                           -1,878.17
Cisco TelePresence System EX90 Videoconferencing Unit               -1,811.08
Martin Yale Chadless Opener Electric Letter Opener                  -1,299.18
Balt Solid Wood Round Tables                                        -1,201.06
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1,148.44
Name: Profit, dtype: float64

Suggested actions to investigate:
- Promote products with consistently high sales and positive profit.
- Review products with repeated negative profit before discontinuing them.
- Examine discount levels and their relationship with profit.
- Investigate regions with relatively low profit.
- Plan inventory and campaigns around observed peak sales periods.


## Phase 15 — Executive Summary

### Findings reported in the assignment document
The supplied assignment report records the following findings for its sample dataset:

- Technology is the strongest category in both sales and profit.
- Furniture has good sales but comparatively weak profitability.
- West is the highest-performing region for sales and profit.
- Central needs improvement.
- Consumer is the largest customer segment by sales contribution.
- Higher discounts are associated with lower profit.
- November and December are peak sales months; January and February are the lowest.

### Recommendations reported in the assignment document
1. Reduce unnecessary discounts.
2. Improve pricing strategy for Furniture.
3. Increase marketing attention in Central and South.
4. Focus on loyal and high-value customers.
5. Prepare inventory for November and December peak demand.

> **Important:** These figures and conclusions are transcribed from the assignment document's sample output. Run the notebook on the exact dataset you use and compare the generated results before submitting. The dataset itself was not included with the uploaded DOCX, so the notebook cannot independently verify those figures yet.

## Final Deliverables Checklist

- [ ] Jupyter Notebook
- [ ] Cleaned Dataset
- [ ] EDA Report
- [ ] Statistical Analysis Report
- [ ] Feature Engineering Report
- [ ] Visualization Dashboard
- [ ] Business Recommendation Report
- [ ] Executive Summary Presentation